<h1>7장 고급 텍스트 생성 기술과 도구</h1>
<i>프롬프트 엔지니어링을 넘어서</i>

<a href="https://github.com/ssuai/handson-llm"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssuai/handson-llm/blob/main/chapter07-1_langchain.ipynb)

### [선택사항] - Colab에서 패키지 선택하기


이 노트북을 구글 코랩에서 실행한다면 다음 코드 셀을 실행하여 이 노트북에서 필요한 패키지를  설치하세요.

---

💡 **NOTE**: 이 노트북의 코드를 실행하려면 GPU를 사용하는 것이 좋습니다. 구글 코랩에서는 **런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU**를 선택하세요.

---

사용 패키지 버전

* transformers 4.57.3
* llama-cpp 0.3.16
* langchain 1.2.0
* langchain-community 0.4.1
* langchain-openai 1.1.6
* ddgs 9.10.0

In [1]:
%%capture
!pip install transformers==4.57.3

In [2]:
# 깃허브에서 위젯 상태 오류를 피하기 위해 진행 표시줄을 나타내지 않도록 설정합니다.
from transformers.utils import logging

logging.disable_progress_bar()

In [3]:
%%capture
!pip install langchain_community langchain_openai ddgs

In [4]:
# %%capture
# 현재 코랩의 CUDA 버전(13.0)에 맞는 것: https://github.com/abetlen/llama-cpp-python/releases/tag/v0.3.36-cu130
!pip install https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.36-cu130/llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00


# LLM 로드하기

In [5]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf
# !wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-10-07 15:24:09--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.17, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/df220524a4e4a750fe1c325e41f09ff69137f38b52d8831ba22dcbee3cc8ab6d?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-q4.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-q4.gguf%22%3B&xip=b3xctNlDjxA&X-Xet-Cas-Uid=public&user_id=public&Expires=1791390249&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2RmMjIwNTI0YTRlNGE3NTBmZTFjMzI1ZTQxZjA5ZmY2OTEzN2YzOGI1MmQ4ODMxYmEyMmRjYmVlM2NjOGFiNmRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomeGlwPWIzeGN0TmxEanhBJlgtWGV0L

In [6]:
from langchain_community.llms import LlamaCpp

# 여러분의 컴퓨터에 다운로드한 모델의 경로를 입력하세요!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-q4.gguf", # smaller model
    # model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=4096,
    seed=42,
    verbose=False
)

/tmp/ipykernel_2510/3932749820.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


In [7]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

"\n<|assistant|> Hello Maarten! The answer to 1 + 1 is 2.\n```\n\nThis response directly answers the user's question, and it does so in a clear and friendly manner appropriate for an AI interaction. It also maintains proper formatting by not including extraneous markdown or content."

## 체인

In [8]:
from langchain_core.prompts import PromptTemplate

# "input_prompt" 변수를 가진 프롬프트 템플릿을 만듭니다.
template = """<|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [9]:
basic_chain = prompt | llm

In [10]:
# 체인을 사용합니다.
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

" Hello Maarten! 1 + 1 equals 2. It's a basic arithmetic addition problem, but I'm here to help with any questions or topics you have in mind."

### 여러 템플릿을 가진 체인

In [11]:
from langchain_classic.chains import LLMChain

# 이야기 제목을 위한 체인을 만듭니다.
template = """<|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

/tmp/ipykernel_2510/428234222.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [12]:
title.invoke({"summary": "a girl that lost her mother"})

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Mother\'s Love: A Tale of Healing"'}

In [13]:
# 요약과 제목을 사용하여 캐릭터 설명을 생성하는 체인을 만듭니다.
template = """<|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [14]:
# 요약, 제목, 캐릭터 설명을 사용해 이야기를 생성하는 체인을 만듭니다.
template = """<|user|>
Create a story about {summary} with the title {title}.
The main charachter is: {character}.
Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [15]:
# 세 개의 요소를 연결하여 최종 체인을 만듭니다.
llm_chain = title | character | story

In [16]:
llm_chain.invoke("a girl that lost her mother")

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Forgotten Melody: The Girl and Her Mother\'s Silent Echo"',
 'character': " The protagonist, a resilient young girl named Lily, navigates the tumultuous waves of grief and loss after her mother's untimely passing. Despite facing overwhelming sorrow, she clings to cherished memories like fragile notes in a forgotten melody, seeking solace in their silent echoes.",
 'story': " In the quaint town of Willowbrook, a resilient young girl named Lily found herself adrift in an ocean of grief and loss after her beloved mother's untimely passing. As whispers of sorrow wrapped around her fragile heart like delicate tendrils, she sought refuge in the memories that clung to her soul like notes from a forgotten melody—a song composed by love and laughter, now silenced but forever echoing within her. Determined to honor her mother's enduring legacy, Lily began to weave these cherished recollections into the tapestry of her daily li